In [ ]:

from pathlib import Path
import csv
from typing import Any, Literal

import matplotlib.pyplot as plt
import numpy as np
import spikeinterface as si
import spikeinterface.extractors as se
from spikeinterface.postprocessing.localization_tools import compute_center_of_mass
from matplotlib.axes import Axes
from matplotlib.colorbar import Colorbar
from matplotlib.collections import PolyCollection, QuadMesh
from matplotlib.figure import Figure
from matplotlib.legend import Legend
from numpy.typing import NDArray
from probeinterface import Probe
from probeinterface.plotting import plot_probe

from Utils.si_utils import validate_data

type FloatArray = NDArray[np.floating[Any]]
type IntArray = NDArray[np.integer[Any]]
type StringArray = NDArray[np.str_]
type StructuredArray = NDArray[np.void]
type AxesArray = NDArray[np.object_]

In [ ]:
plot_colors: dict[str, str] = {
    'ink': '#172033',
    'muted': '#667085',
    'border': '#CBD5E1',
    'grid': '#E2E8F0',
    'contact': '#D8DEE8',
    'probe': '#F8FAFC',
}
unit_colors: tuple[str, ...] = ('#2563EB', '#D97706', '#0F766E', '#7C3AED', '#BE123C', '#4F46E5')

# Force a light plotting theme even when Jupyter itself uses dark mode.
plt.style.use('default')
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
    'savefig.transparent': False,
    'text.color': plot_colors['ink'],
    'axes.labelcolor': plot_colors['ink'],
    'axes.edgecolor': plot_colors['border'],
    'axes.titlecolor': plot_colors['ink'],
    'xtick.color': plot_colors['muted'],
    'ytick.color': plot_colors['muted'],
    'grid.color': plot_colors['grid'],
    'grid.linewidth': 0.7,
    'grid.alpha': 0.8,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'image.interpolation': 'none',
    'path.simplify': False,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'figure.dpi': 110,
    'savefig.dpi': 200,
})


def style_axes(ax: Axes, grid: bool = False) -> None:
    ax.set_facecolor('white')
    ax.tick_params(colors=plot_colors['muted'], labelsize=9)
    for spine in ax.spines.values():
        spine.set_color(plot_colors['border'])
    ax.grid(grid)
    ax.set_axisbelow(True)


def make_panel_grid(
        panel_count: int,
        panel_width: float,
        panel_height: float,
        max_columns: int = 3,
        sharex: bool = False,
        sharey: bool = False,
) -> tuple[Figure, AxesArray]:
    if panel_count < 1:
        raise ValueError('panel_count must be positive.')
    column_count: int = min(max_columns, panel_count)
    row_count: int = (panel_count + column_count - 1) // column_count
    fig: Figure
    axes: AxesArray
    fig, axes = plt.subplots(
        row_count,
        column_count,
        figsize=(panel_width * column_count, panel_height * row_count),
        squeeze=False,
        sharex=sharex,
        sharey=sharey,
        constrained_layout=True,
    )
    return fig, np.asarray(axes, dtype=object)


def hide_unused_axes(axes: AxesArray, used_count: int) -> None:
    for unused_ax in axes.ravel()[used_count:]:
        unused_ax.set_visible(False)

In [ ]:
def plot_probe_geometry(plotting_probe):
    fig, ax = plt.subplots(figsize=(6.4, 8.8), constrained_layout=True)
    plot_probe(
        plotting_probe,
        ax=ax,
        title=False,
        contacts_colors=plot_colors['contact'],
        contact_kwargs={'alpha': 1.0, 'edgecolor': plot_colors['border'], 'lw': 0.25},
        probe_shape_kwargs={'facecolor': plot_colors['probe'], 'edgecolor': plot_colors['border'], 'lw': 0.8},
    )
    style_axes(ax)
    ax.set_title(f'{session_dir.name} · Probe{probe_name} geometry', loc='left', fontweight='bold')
    ax.set_xlabel('Probe x (µm)')
    ax.set_ylabel('Probe y (µm)')
    if save_figures:
        fig.savefig(output_dir / 'probe_geometry.png', dpi=200, bbox_inches='tight')
    plt.show()